# E1.2 — Trajectory Extension (BATCH) — Paper 1 extension

Runs a user-defined list of `(system, replica)` pairs **in succession, unattended** — built
for leaving overnight. Same protocol as the single-run notebook (frame-0 start coordinates,
`LangevinIntegrator`, PME 1.0 nm, HBonds, MC barostat, 1 ps saving); the only difference is
the driver loops over a run list and each run self-verifies.

**Before you start an unattended batch — non-negotiable, because there is no checkpoint:**
- On AC power.
- `caffeinate -is` running in a spare terminal, lid open. A mid-run sleep loses that run with
  nothing to resume from.

**What each run does:** builds from frame 0 of `{system}_prod.dcd` (prmtop order), guards the
start energy, runs 100 ns, saves DCD / final state / log / metadata, then verifies the DCD
(frame count and a non-NaN last frame) and records a pass/fail. A failure in one run does not
stop the others.

**Ordering:** keep the list to the next *first* replicates until all nine are done and τ_int
is re-checked — not rep2/rep3 of a system already run. Default below is the two remaining
water first-replicates.

## 1 — Imports
**Out:** namespace; prints versions and platforms.

In [1]:
import os, sys, time, json, hashlib, subprocess, gc, traceback
import platform as _platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import openmm as mm
import openmm.app as app
import openmm.unit as unit
from openmm import LangevinIntegrator, MonteCarloBarostat, Platform
import mdtraj as md
from mdtraj.formats import DCDTrajectoryFile

print(f"OpenMM {mm.__version__} | MDTraj {md.__version__}")
print("Platforms:", [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())])

OpenMM 8.5.1 | MDTraj 1.11.1
Platforms: ['Reference', 'CPU', 'OpenCL']


## 2 — Fixed protocol constants
Carried forward from the original production. **Do not edit between runs.**

In [2]:
SYSTEMS = [
    "GGE_water", "CME_water", "YIY_water",
    "GGE_reline", "CME_reline", "YIY_reline",
    "GGE_glyceline", "CME_glyceline", "YIY_glyceline",
]

TEMPERATURE = 300 * unit.kelvin
PRESSURE    = 1 * unit.bar
FRICTION    = 1.0 / unit.picosecond
TIMESTEP    = 2.0 * unit.femtoseconds
PME_CUTOFF  = 1.0 * unit.nanometers

SAVE_INTERVAL   = 500      # 1 ps
LOG_INTERVAL    = 5000     # 10 ps
STDOUT_INTERVAL = 100000   # 200 ps

# Measured sustained rate from GGE_water rep1 (609 ns/day); cold benchmark was 710.
SUSTAINED_NS_PER_DAY = 610.0

## 3 — Run list  *(the only cell to edit per batch)*
List the `(system, replica)` pairs to run, in order. They execute top to bottom.
**Out:** `RUN_LIST` and shared configuration.

In [3]:
RUN_LIST = [
    ("GGE_glyceline", 1),
    ("CME_glyceline", 1),
]

PROJECT_DIR = Path("~/des-peptide-study").expanduser()
SYSTEMS_DIR = PROJECT_DIR / "systems"
OUTPUT_DIR  = PROJECT_DIR / "extension" / "trajectories"

NS       = 100.0
PLATFORM = "OpenCL"

## 4 — Helpers
Seed derivation, path resolution (subdir or flat), and the frame-0 coordinate loader.

In [4]:
def derive_seed(system, replica):
    digest = hashlib.sha256(f"{system}_rep{replica}".encode()).digest()
    return int.from_bytes(digest[:4], "big") % (2 ** 31 - 2) + 1

def resolve(system, name):
    for cand in (SYSTEMS_DIR / system / name, SYSTEMS_DIR / name):
        if cand.exists():
            return cand
    return None

def load_start(prmtop_path, prod_dcd_path):
    t0 = md.load_frame(str(prod_dcd_path), 0, top=str(prmtop_path))
    return t0.openmm_positions(0), t0.openmm_boxes(0)

## 5 — Per-run function
Encapsulates build → guard → run → save → verify for one `(system, replica)`. Returns a
result dict; never raises (errors are caught and recorded so the batch continues). Releases
the GPU context before returning.

In [5]:
def run_one(system, replica):
    result = {"system": system, "replica": replica, "status": "started"}
    sim = None
    try:
        seed = derive_seed(system, replica)
        prmtop_path = resolve(system, f"{system}.prmtop")
        prod_dcd    = resolve(system, f"{system}_prod.dcd")
        assert prmtop_path is not None, f"{system}.prmtop not found"
        assert prod_dcd is not None,    f"{system}_prod.dcd not found"

        tag = f"{system}_rep{replica}"
        run_dir = OUTPUT_DIR / tag
        run_dir.mkdir(parents=True, exist_ok=True)
        dcd       = run_dir / f"{tag}.dcd"
        state_xml = run_dir / f"{tag}_final_state.xml"
        logf      = run_dir / f"{tag}_production.log"
        metaf     = run_dir / f"{tag}_metadata.json"
        total_steps = int(NS * 1e6 / TIMESTEP.value_in_unit(unit.femtoseconds))

        prmtop = app.AmberPrmtopFile(str(prmtop_path))
        positions, box = load_start(prmtop_path, prod_dcd)

        system_obj = prmtop.createSystem(nonbondedMethod=app.PME,
                                         nonbondedCutoff=PME_CUTOFF, constraints=app.HBonds)
        system_obj.setDefaultPeriodicBoxVectors(*box)
        baro = MonteCarloBarostat(PRESSURE, TEMPERATURE); baro.setRandomNumberSeed(seed)
        system_obj.addForce(baro)
        integ = LangevinIntegrator(TEMPERATURE, FRICTION, TIMESTEP); integ.setRandomNumberSeed(seed)

        sim = app.Simulation(prmtop.topology, system_obj, integ, Platform.getPlatformByName(PLATFORM))
        sim.context.setPositions(positions)
        sim.context.setPeriodicBoxVectors(*box)
        sim.context.setVelocitiesToTemperature(TEMPERATURE, seed)

        start_pe = sim.context.getState(getEnergy=True).getPotentialEnergy()
        assert abs(start_pe.value_in_unit(unit.kilojoules_per_mole)) < 1e7, f"start PE {start_pe} non-physical"

        sim.reporters.append(app.DCDReporter(str(dcd), SAVE_INTERVAL))
        sim.reporters.append(app.StateDataReporter(str(logf), LOG_INTERVAL,
            step=True, time=True, potentialEnergy=True, temperature=True, density=True, speed=True))
        sim.reporters.append(app.StateDataReporter(sys.stdout, STDOUT_INTERVAL,
            step=True, time=True, temperature=True, speed=True, remainingTime=True, totalSteps=total_steps))

        started = datetime.now(timezone.utc); _t = time.time()
        sim.step(total_steps)
        wall = time.time() - _t; finished = datetime.now(timezone.utc)
        sim.saveState(str(state_xml))

        with DCDTrajectoryFile(str(dcd)) as f:
            n_frames = len(f)
        expected = total_steps // SAVE_INTERVAL
        last = md.load_frame(str(dcd), n_frames - 1, top=str(prmtop_path))
        last_nan = bool(np.isnan(last.xyz).any())
        verified = (n_frames == expected) and (not last_nan)

        achieved = (total_steps * TIMESTEP.value_in_unit(unit.nanoseconds)) / (wall / 86400.0)
        metadata = {
            "phase": "E1.2", "system": system, "replica": replica, "seed": seed,
            "ns": NS, "total_steps": total_steps, "save_interval_steps": SAVE_INTERVAL,
            "integrator": "LangevinIntegrator", "temperature_K": 300,
            "friction_per_ps": 1.0, "timestep_fs": 2.0, "barostat": "MonteCarlo (1 bar)",
            "nonbonded": "PME, 1.0 nm cutoff, HBonds constraints",
            "platform": PLATFORM, "precision": "default",
            "topology": str(prmtop_path), "start_coordinates": f"{prod_dcd} (frame 0)",
            "started_utc": started.isoformat(), "finished_utc": finished.isoformat(),
            "wall_seconds": round(wall, 1), "achieved_ns_per_day": round(achieved, 1),
            "frames_written": n_frames, "frames_expected": expected,
            "verified": verified,
            "openmm_version": mm.__version__, "mdtraj_version": md.__version__,
            "python_version": _platform.python_version(), "host": _platform.node(),
        }
        metaf.write_text(json.dumps(metadata, indent=2))

        result.update(status=("ok" if verified else "VERIFY_FAILED"),
                      frames=n_frames, expected=expected, last_nan=last_nan,
                      ns_per_day=round(achieved, 1), wall_h=round(wall / 3600, 2))
    except Exception as e:
        result.update(status="ERROR", error=repr(e))
        traceback.print_exc()
    finally:
        if sim is not None:
            del sim
        gc.collect()
    return result

## 6 — Batch pre-flight (compute + storage)
Estimates total wall time and storage for the whole list **before** anything runs. Starts nothing.

In [6]:
print("=" * 70)
print(f"{'system':<16}{'rep':>4}{'atoms':>8}{'est h':>8}{'est GB':>9}")
print("-" * 70)
tot_h = tot_gb = 0.0
for system, replica in RUN_LIST:
    p = resolve(system, f"{system}.prmtop")
    d = resolve(system, f"{system}_prod.dcd")
    flag = "" if (p and d) else "  <-- INPUTS MISSING"
    n_atoms = app.AmberPrmtopFile(str(p)).topology.getNumAtoms() if p else 0
    n_frames = int(NS * 1e6 / TIMESTEP.value_in_unit(unit.femtoseconds)) // SAVE_INTERVAL
    est_h = NS / SUSTAINED_NS_PER_DAY * 24.0
    est_gb = n_frames * (n_atoms * 12 + 80) / 1e9
    tot_h += est_h; tot_gb += est_gb
    print(f"{system:<16}{replica:>4}{n_atoms:>8}{est_h:>8.2f}{est_gb:>9.2f}{flag}")
print("-" * 70)
print(f"{'TOTAL':<28}{tot_h:>8.2f}{tot_gb:>9.2f}   (@ {SUSTAINED_NS_PER_DAY:g} ns/day, measured)")
print("=" * 70)
print("Confirm AC power + caffeinate -is + lid open before running the driver cell.")

system           rep   atoms   est h   est GB
----------------------------------------------------------------------
GGE_glyceline      1    2894    3.93     3.48
CME_glyceline      1    2925    3.93     3.52
----------------------------------------------------------------------
TOTAL                           7.87     7.00   (@ 610 ns/day, measured)
Confirm AC power + caffeinate -is + lid open before running the driver cell.


## 7 — Driver loop
Runs each pair in sequence. Prints a banner per run; a failure is recorded and the batch
continues. Leave it running — closing the browser tab does not stop the kernel.
**Out:** `RESULTS`.

In [7]:
RESULTS = []
batch_start = datetime.now()
print(f"BATCH START {batch_start:%Y-%m-%d %H:%M:%S} — {len(RUN_LIST)} run(s)\n")

for system, replica in RUN_LIST:
    print("=" * 64)
    print(f">>> {system} rep{replica}   start {datetime.now():%H:%M:%S}")
    print("=" * 64)
    r = run_one(system, replica)
    RESULTS.append(r)
    if r["status"] == "ERROR":
        print(f"<<< {system} rep{replica}: ERROR — {r['error']}\n")
    else:
        print(f"<<< {system} rep{replica}: {r['status']}  "
              f"{r.get('ns_per_day')} ns/day, {r.get('frames')}/{r.get('expected')} frames\n")

print(f"BATCH END {datetime.now():%Y-%m-%d %H:%M:%S} "
      f"(elapsed {(datetime.now()-batch_start).total_seconds()/3600:.2f} h)")

BATCH START 2026-06-07 11:01:12 — 2 run(s)

>>> GGE_glyceline rep1   start 11:01:12
#"Step","Time (ps)","Temperature (K)","Speed (ns/day)","Time Remaining"
100000,200.00000000022686,294.43050226217906,0,--
200000,400.00000000118183,302.29550458952536,692,3:27:23
300000,599.9999999996356,291.53058379808175,691,3:27:09
400000,799.9999999949063,293.9528581438453,690,3:27:06
500000,999.9999999901769,301.2818191069343,689,3:26:58
600000,1199.9999999854476,293.05656798769263,688,3:26:44
700000,1399.9999999807183,302.15490929515784,688,3:26:24
800000,1599.9999999759889,290.9234113709173,688,3:26:00
900000,1799.9999999712595,296.41901279653575,688,3:25:38
1000000,1999.9999999665301,302.9403745418545,688,3:25:15
1100000,2199.9999999618008,291.6524563113135,687,3:24:53
1200000,2399.9999999570714,297.2722560614849,687,3:24:27
1300000,2599.999999952342,301.20516424213923,687,3:24:03
1400000,2799.9999999476127,303.73551128729525,687,3:23:41
1500000,2999.9999999428833,296.5942626795894,687,3:23:18
1

## 8 — Batch summary
A glance-able pass/fail table for when you return. `ok` means complete frame count and a
non-NaN last frame; anything else needs a look. Commit the metadata and logs of the `ok`
runs (DCDs stay local).

In [8]:
print(f"{'system':<16}{'rep':>4}{'status':>14}{'ns/day':>9}{'frames':>14}{'wall h':>8}")
print("-" * 66)
for r in RESULTS:
    frames = f"{r.get('frames','-')}/{r.get('expected','-')}" if r['status'] != 'ERROR' else '-'
    print(f"{r['system']:<16}{r['replica']:>4}{r['status']:>14}"
          f"{str(r.get('ns_per_day','-')):>9}{frames:>14}{str(r.get('wall_h','-')):>8}")
print("-" * 66)
n_ok = sum(1 for r in RESULTS if r['status'] == 'ok')
print(f"{n_ok}/{len(RESULTS)} verified ok")

system           rep        status   ns/day        frames  wall h
------------------------------------------------------------------
GGE_glyceline      1            ok    592.7 100000/100000    4.05
CME_glyceline      1            ok    461.8 100000/100000     5.2
------------------------------------------------------------------
2/2 verified ok
